# SmartAutoDJ — Generative Bridge (MusicGen-Style) — ⚠️ LEGACY COLAB FALLBACK

> **Primary path is now Modal, not Colab.** Generation moved to `infra/modal_bridge.py`
> (serverless CUDA, deps baked into an image once — no per-session reinstall). Use:
> ```bash
> pip install -e ".[gen]" && modal setup && modal deploy infra/modal_bridge.py
> python -m smartautodj.pipeline --song-a A.mp3 --song-b B.mp3 --tier 3 --generate
> ```
> Keep this notebook only as a manual fallback if you can't use Modal. See CLAUDE.md §5.

Generate the **tier-3 additive bridge** — a short riser/sweep that mixes *under*
the algorithmic transition. The clip is conditioned on **both** a text prompt
*and* a reference audio excerpt taken from the A→B boundary, so it inherits the
key/timbre of the actual transition (MusicGen-Style audio conditioning).

**Workflow** (Env B, GPU — keep separate from the local analysis env, see CLAUDE.md §5):
1. Run the local pipeline at tier 3 once. With no AI clip present it falls back
   to the procedural riser and writes two conditioning artifacts next to the
   output WAV:
   * `*_boundary_ref.wav` — the reference excerpt to condition on.
   * `plan.bridge.prompt` in the JSON sidecar — the auto-built text prompt.
2. Upload the reference here, paste the prompt, generate.
3. Download the result to `assets/generated/<pair>_bridge.wav` in the repo.
4. Re-run the local pipeline at tier 3 — `resolve_bridge` now loads your clip
   instead of the placeholder.

> Runtime → Change runtime type → **GPU** before running.
> **API caveat:** the `generate_with_chroma(descriptions=..., melody=..., sr=...)` call below
> matches the audiocraft docs at time of writing, but the exact signature has changed across
> audiocraft versions (positional args / `melody_sample_rate`, and `musicgen-style` style-
> conditioning may differ). If it errors, check `help(model.generate_with_chroma)` for the
> version that installs in your Colab runtime and adjust. The local pipeline does not depend
> on this notebook — it just consumes the resulting WAV.

In [ ]:
# 0. Confirm a GPU is attached (MusicGen needs CUDA; CPU is far too slow).
import torch
assert torch.cuda.is_available(), 'No GPU — set Runtime > Change runtime type > GPU.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 1. Install audiocraft (MusicGen / MusicGen-Style). torch is preinstalled on Colab.
#    audiocraft pins its own deps; install into the fresh Colab runtime only.
%pip install -q audiocraft

In [ ]:
# 2. Upload the boundary-reference WAV exported by the local pipeline
#    (outputs/<pair>__tier3_boundary_ref.wav).
from google.colab import files
uploaded = files.upload()
REF_PATH = next(iter(uploaded))
print('reference:', REF_PATH)

In [ ]:
# 3. Settings — paste the prompt from the JSON sidecar (plan.bridge.prompt),
#    set the pair name (used for the output filename) and the overlap duration.
PROMPT = '124 BPM electronic transition riser in A minor, smoothly building energy into a drop, atmospheric sweep and white-noise rise, no lead melody, about 14 seconds'
PAIR = 'songA__songB'   # -> assets/generated/<PAIR>_bridge.wav
DURATION = 14.0          # seconds; match plan.overlap_sec from the sidecar
MODEL_ID = 'facebook/musicgen-style'  # audio+text conditioned; or 'facebook/musicgen-melody'

In [ ]:
# 4. Load the model and configure text + audio (style) conditioning.
import torchaudio
from audiocraft.models import MusicGen

model = MusicGen.get_pretrained(MODEL_ID)
model.set_generation_params(
    duration=DURATION,
    use_sampling=True,
    top_k=250,
    cfg_coef=3.0,
    cfg_coef_beta=5.0,   # required for joint text+style conditioning (musicgen-style)
)
if MODEL_ID.endswith('musicgen-style'):
    # eval_q: style fidelity (lower = looser); excerpt_length: seconds of ref used.
    model.set_style_conditioner_params(eval_q=1, excerpt_length=3.0)

In [ ]:
# 5. Generate the bridge conditioned on the reference excerpt + the text prompt.
ref, sr = torchaudio.load(REF_PATH)
ref = ref.mean(0, keepdim=True)  # mono
wav = model.generate_with_chroma(
    descriptions=[PROMPT],
    melody=ref[None].expand(1, -1, -1),
    sr=sr,
)
print('generated', wav.shape, '@', model.sample_rate, 'Hz')

In [ ]:
# 6. Preview, then save to the conventional repo path and download.
from audiocraft.data.audio import audio_write
from IPython.display import Audio, display

out_stem = f'{PAIR}_bridge'
audio_write(out_stem, wav[0].cpu(), model.sample_rate,
            strategy='loudness', loudness_compressor=True)
out_file = out_stem + '.wav'
display(Audio(out_file))

from google.colab import files as _files
print('Download, then place in the repo at assets/generated/' + out_file)
_files.download(out_file)

## Back in the local repo

```bash
mv ~/Downloads/<PAIR>_bridge.wav assets/generated/
python -m smartautodj.pipeline --song-a A.wav --song-b B.wav --tier 3   # loads your clip
```

`resolve_bridge` finds `assets/generated/<pair>_bridge.wav` automatically (or pass
`--bridge-clip path.wav`). The clip flows through `prepare_clip` (trim → normalize
→ length-match) and mixes additively under the transition, gain-staged so it sits
below the mix. The procedural riser remains the fallback if the file is absent.